# 05 – t-tests & Chi-squared Tests

Author: Joe  
Project: AI Foundations – Probability & Statistics

This notebook extends the inference tools from 04 into the classic tests you meet in
intro stats / MATH4002:

- **t-tests** for means (unknown variance)
- **Chi-squared tests** for categorical data

We'll stick to the core exam-style cases:

- One-sample t-test
- Two-sample t-test (independent samples)
- Paired t-test
- Chi-squared goodness-of-fit
- Chi-squared test of independence (contingency tables)

Conceptually you'll still use R / tables for exact numbers, but here we build intuition
and simulate p-values so you can *see* what the tests are doing.

## Contents
1. [Setup](#1-setup)
2. [When to use t vs z](#2-when-to-use-t-vs-z)
3. [One-sample t-test](#3-one-sample-t-test)
4. [Two-sample t-test (independent)](#4-two-sample-t-test-independent)
5. [Paired t-test](#5-paired-t-test)
6. [Chi-squared: Goodness-of-fit](#6-chi-squared-goodness-of-fit)
7. [Chi-squared: Test of independence](#7-chi-squared-test-of-independence)
8. [Practice / TODOs (MATH4002 alignment)](#8-practice--todos-math4002-alignment)


## 1. Setup

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

np.set_printoptions(precision=4, suppress=True)

def normal_cdf(z: float) -> float:
    """Standard normal CDF using the error function."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

def approx_t_p_value(t_stat: float, df: int, alternative: str = 'two-sided', n_sim: int = 100000) -> float:
    """Approximate p-value for a t-statistic using Monte Carlo simulation.

    This is for intuition / teaching. For exams, you'd usually use R's pt()/t.test() or tables.
    """
    sims = np.random.standard_t(df, size=n_sim)
    if alternative == 'two-sided':
        return np.mean(np.abs(sims) >= abs(t_stat))
    elif alternative == 'greater':
        return np.mean(sims >= t_stat)
    elif alternative == 'less':
        return np.mean(sims <= t_stat)
    else:
        raise ValueError("alternative must be 'two-sided', 'greater', or 'less'")

def approx_chi2_p_value(chi2_stat: float, df: int, alternative: str = 'greater', n_sim: int = 100000) -> float:
    """Approximate upper-tail p-value for a chi-squared statistic via simulation.

    Most chi-squared tests are right-tailed (large chi2 = more evidence against H0).
    """
    sims = np.random.chisquare(df, size=n_sim)
    if alternative != 'greater':
        raise ValueError("Chi-squared tests are usually 'greater' (right-tailed).")
    return np.mean(sims >= chi2_stat)


---
## 2. When to use t vs z

Very exam-relevant distinction:

- Use **z** when:
  - population standard deviation $\sigma$ is known, *and/or*
  - sample size is large and you're explicitly told to use a z-approximation.

- Use **t** when:
  - $\sigma$ is **unknown**,
  - you estimate it with sample standard deviation $s$,
  - and data are (approximately) Normal or $n$ is not tiny.

t-distributions are like Normal(0,1) but with **heavier tails**, especially for small
degrees of freedom. They gradually converge to Normal as df increases.

In [ ]:
# Visual comparison: t with different df vs standard normal
x = np.linspace(-4, 4, 400)
pdf_normal = (1 / math.sqrt(2 * math.pi)) * np.exp(-x**2 / 2)

plt.plot(x, pdf_normal, label='Normal(0,1)')
for df in [2, 5, 10]:
    sims = np.random.standard_t(df, size=200000)
    # Approximate pdf by histogram
    hist_y, hist_x = np.histogram(sims, bins=80, range=(-4, 4), density=True)
    centers = 0.5 * (hist_x[1:] + hist_x[:-1])
    plt.plot(centers, hist_y, label=f't(df={df})')

plt.xlabel('x')
plt.ylabel('Density')
plt.title('t-distributions vs Normal(0,1) (simulated)')
plt.legend()
plt.show()

---
## 3. One-sample t-test

Scenario (very common in assignments):

- We have a sample $X_1, \dots, X_n$ from a Normal population with **unknown** mean $\mu$ and unknown $\sigma$.
- We want to test
  $$H_0: \mu = \mu_0 \quad \text{vs} \quad H_1: \mu \ne \mu_0$$
  (or one-sided versions).

Test statistic:
$$t = \frac{\bar{X} - \mu_0}{s / \sqrt{n}},$$
where $s$ is the sample standard deviation.

Under $H_0$, if data are Normal, this follows a **t-distribution with $n-1$ degrees of freedom**.

In R you'd usually do something like:

```r
t.test(x, mu = mu0)
```

In [ ]:
def one_sample_t_test(sample: np.ndarray, mu0: float, alternative: str = 'two-sided'):
    """One-sample t-test.

    Returns: (x_bar, t_stat, df, p_value_approx)
    """
    n = len(sample)
    x_bar = float(sample.mean())
    s = float(sample.std(ddof=1))
    df = n - 1
    se = s / math.sqrt(n)
    t_stat = (x_bar - mu0) / se
    p_value = approx_t_p_value(t_stat, df=df, alternative=alternative)
    return x_bar, t_stat, df, p_value

# Example: sample of exam scores, test H0: mu = 60
np.random.seed(0)
mu_true = 62
sigma_true = 10
sample = np.random.normal(loc=mu_true, scale=sigma_true, size=25)

x_bar, t_stat, df, p_val = one_sample_t_test(sample, mu0=60, alternative='two-sided')
x_bar, t_stat, df, p_val

---
## 4. Two-sample t-test (independent)

Scenario: compare means of two independent groups (e.g. treatment vs control).

Let group 1 have $n_1$ observations, mean $\bar{X}_1$, variance $s_1^2$.
Let group 2 have $n_2$ observations, mean $\bar{X}_2$, variance $s_2^2$.

We often test
$$H_0: \mu_1 = \mu_2 \quad \text{vs} \quad H_1: \mu_1 \ne \mu_2.$$

There are two main flavours you see:

- **Pooled-variance (equal variances) t-test** – assumes $\sigma_1^2 = \sigma_2^2$.
- **Welch's t-test** – does not assume equal variances (this is what R's `t.test()` does by default).

Here we implement the **Welch** version, which is safer in practice.

In [ ]:
def two_sample_t_test_welch(x: np.ndarray, y: np.ndarray, alternative: str = 'two-sided'):
    """Welch two-sample t-test (independent samples).

    Returns: (mean_x, mean_y, t_stat, df_approx, p_value_approx)
    """
    n1, n2 = len(x), len(y)
    mean1, mean2 = float(x.mean()), float(y.mean())
    s1_sq, s2_sq = float(x.var(ddof=1)), float(y.var(ddof=1))

    se_sq = s1_sq / n1 + s2_sq / n2
    t_stat = (mean1 - mean2) / math.sqrt(se_sq)

    # Welch–Satterthwaite df approximation
    df_num = se_sq ** 2
    df_den = (s1_sq**2 / (n1**2 * (n1 - 1))) + (s2_sq**2 / (n2**2 * (n2 - 1)))
    df = df_num / df_den

    p_value = approx_t_p_value(t_stat, df=int(round(df)), alternative=alternative)
    return mean1, mean2, t_stat, df, p_value

# Example: group A vs group B
np.random.seed(1)
group_A = np.random.normal(loc=100, scale=15, size=30)
group_B = np.random.normal(loc=92, scale=15, size=28)

mean_A, mean_B, t_stat_AB, df_AB, p_val_AB = two_sample_t_test_welch(group_A, group_B)
mean_A, mean_B, t_stat_AB, df_AB, p_val_AB

---
## 5. Paired t-test

Scenario: **paired / matched** data – typical MATH4002-style example:

- Before-and-after measurements on the same subjects (e.g. blood pressure pre/post treatment).
- Matched pairs (e.g. twins, or matched by age/gender/etc.).

Instead of comparing two independent samples, you:

1. Compute differences $D_i = X_{i,\text{after}} - X_{i,\text{before}}$.
2. Do a **one-sample t-test on the differences**, with $H_0: \mu_D = 0$.

The test statistic is the usual one-sample t using the $D_i$'s.

In [ ]:
def paired_t_test(before: np.ndarray, after: np.ndarray, alternative: str = 'two-sided'):
    """Paired t-test (one-sample t on differences)."""
    if len(before) != len(after):
        raise ValueError('before and after must have same length')
    diff = after - before
    return one_sample_t_test(diff, mu0=0.0, alternative=alternative)

# Example: before vs after scores
np.random.seed(2)
n = 20
before = np.random.normal(loc=50, scale=5, size=n)
improvement = np.random.normal(loc=3, scale=2, size=n)
after = before + improvement

x_bar_diff, t_stat_diff, df_diff, p_val_diff = paired_t_test(before, after)
x_bar_diff, t_stat_diff, df_diff, p_val_diff

---
## 6. Chi-squared: Goodness-of-fit

Goodness-of-fit tests compare **observed counts** in categories to **expected counts** under some
theoretical distribution.

Example MATH4002-style situation:

- You roll a die 120 times and observe counts in faces 1–6.
- $H_0$: die is fair (each face has probability 1/6).
- $H_1$: die is not fair.

Test statistic:
$$\chi^2 = \sum_{i=1}^k \frac{(O_i - E_i)^2}{E_i},$$
where $O_i$ are observed counts and $E_i$ are expected counts under $H_0$.

Under $H_0$ (and with reasonable expected counts), this approximately follows a chi-squared
distribution with $k - 1 - m$ degrees of freedom, where $m$ is the number of estimated parameters
in the model for the probabilities (often $m=0$ in simple examples).

In [ ]:
def chi_squared_gof(observed: np.ndarray, expected_probs: np.ndarray):
    """Chi-squared goodness-of-fit statistic.

    observed: array of observed counts O_i
    expected_probs: array of probabilities p_i under H0 (sum to 1)
    """
    observed = np.array(observed, dtype=float)
    expected_probs = np.array(expected_probs, dtype=float)
    if not np.isclose(expected_probs.sum(), 1.0):
        raise ValueError('expected_probs must sum to 1')
    n = observed.sum()
    expected = n * expected_probs
    chi2 = np.sum((observed - expected) ** 2 / expected)
    df = len(observed) - 1  # simple case: no parameters estimated
    p_value = approx_chi2_p_value(chi2, df=df)
    return chi2, df, p_value, expected

# Example: die rolled 120 times
observed_counts = np.array([16, 18, 17, 25, 23, 21])
expected_probs = np.ones(6) / 6

chi2_stat, df_gof, p_val_gof, expected_counts = chi_squared_gof(observed_counts, expected_probs)
chi2_stat, df_gof, p_val_gof, expected_counts

In [ ]:
# Visualise observed vs expected counts for the die example
categories = np.arange(1, 7)
width = 0.35

plt.bar(categories - width/2, observed_counts, width=width, label='Observed')
plt.bar(categories + width/2, expected_counts, width=width, label='Expected')
plt.xlabel('Face')
plt.ylabel('Count')
plt.title('Chi-squared GOF example: die rolls')
plt.legend()
plt.show()

---
## 7. Chi-squared: Test of independence

This is for contingency tables (cross-tabulating two categorical variables).

Example: survey of people by **smoker/non-smoker** and **exercise / no exercise**.

- $H_0$: the two variables are independent.
- $H_1$: they are not independent.

Steps:

1. Put observed counts into a $r \times c$ table.
2. Compute row totals, column totals, and grand total $N$.
3. Expected counts under independence:
   $$E_{ij} = \frac{(\text{row } i \text{ total}) (\text{column } j \text{ total})}{N}.$$
4. Compute
   $$\chi^2 = \sum_{i=1}^r \sum_{j=1}^c \frac{(O_{ij} - E_{ij})^2}{E_{ij}}.$$
5. Under $H_0$, this is approximately chi-squared with $(r-1)(c-1)$ degrees of freedom.

In [ ]:
def chi_squared_independence(table: np.ndarray):
    """Chi-squared test of independence for an r x c contingency table.

    table: 2D array of observed counts O_{ij}
    """
    table = np.array(table, dtype=float)
    row_totals = table.sum(axis=1, keepdims=True)
    col_totals = table.sum(axis=0, keepdims=True)
    grand_total = table.sum()

    expected = row_totals @ col_totals / grand_total
    chi2 = np.sum((table - expected) ** 2 / expected)
    r, c = table.shape
    df = (r - 1) * (c - 1)
    p_value = approx_chi2_p_value(chi2, df=df)
    return chi2, df, p_value, expected

# Example 2x3 table (made-up data)
table = np.array([
    [20, 30, 25],  # group 1
    [22, 28, 35],  # group 2
])

chi2_ind, df_ind, p_val_ind, expected_ind = chi_squared_independence(table)
chi2_ind, df_ind, p_val_ind, expected_ind

In [ ]:
# Visualise observed vs expected as side-by-side bars for each cell
r, c = table.shape
indices = np.arange(r * c)
obs_flat = table.flatten()
exp_flat = expected_ind.flatten()
width = 0.35

plt.bar(indices - width/2, obs_flat, width=width, label='Observed')
plt.bar(indices + width/2, exp_flat, width=width, label='Expected')
plt.xlabel('Cell index (flattened)')
plt.ylabel('Count')
plt.title('Chi-squared independence example: observed vs expected')
plt.legend()
plt.show()

---
## 8. Practice / TODOs (MATH4002 alignment)

This notebook is designed to mirror the kinds of questions you see in TU Dublin's
Probability & Statistical Inference module when you hit **t-tests** and
**Chi-squared tests**.

### 8.1 t-test style questions

- One-sample t on a sample of measurements ("is the mean equal to 50?").
- Two-sample t for comparing two independent groups (e.g. two teaching methods).
- Paired t for before/after data.

You can follow this pattern in a new cell:

```python
# Example skeleton for a one-sample t question
data = np.array([...])  # your sample
x_bar, t_stat, df, p_val = one_sample_t_test(data, mu0=..., alternative='two-sided')
x_bar, t_stat, df, p_val
```

### 8.2 Chi-squared style questions

- Goodness-of-fit: are observed category counts consistent with a claimed distribution?
- Independence: is there an association between two categorical variables?

Skeletons:

```python
# Goodness-of-fit
observed = np.array([...])
expected_probs = np.array([...])  # must sum to 1
chi2, df, p_val, expected = chi_squared_gof(observed, expected_probs)
chi2, df, p_val

# Independence
table = np.array([
    [...],
    [...],
])
chi2, df, p_val, expected = chi_squared_independence(table)
chi2, df, p_val
```

### 8.3 Link back to your R workflow

In MATH4002 you'll typically:

- Use R for exact p-values and critical values (`t.test`, `chisq.test`).
- Use by-hand formulas for showing working and understanding.

This notebook is the **bridge**: you can recreate any tutorial question here,
check the mechanics of the test statistic, and use simulations to build your
intuition for what the p-values are doing.